# HRTF Personalization — Full Pipeline

Runs the complete pipeline:
1. Install dependencies and clone the repo
2. Fetch the HRTFCNN dataset
3. Prepare training samples from CIPIC
4. Train the conditional model
5. Evaluate
6. Predict a personalized SOFA file from your ear photos
7. Inspect and download the result

**Runtime:** Set `Runtime > Change runtime type > T4 GPU` before starting.

## 1 — Setup

In [ ]:
import os

REPO_URL = 'https://github.com/ctruett/hrtf_personalization.git'
REPO_DIR = '/content/hrtf_personalization'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -e . --quiet

In [ ]:
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))

### Optional — Mount Google Drive
Skip if you don't need to persist checkpoints across sessions.

In [ ]:
USE_DRIVE = False  # set True to save checkpoints to Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = '/content/drive/MyDrive/hrtf_checkpoints'
else:
    CHECKPOINT_DIR = f'{REPO_DIR}/artifacts/checkpoints'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {CHECKPOINT_DIR}')

## 2 — Fetch HRTFCNN Dataset

Downloads CIPIC SOFA files, anthropometry data, and ear photos (~600 MB).

In [ ]:
import yaml

HRTFCNN_ROOT = '/tmp/HRTFCNN'
PREPARED_ROOT = f'{REPO_DIR}/artifacts/prepared'

dataset_config = {
    'dataset': {
        'source_layout': 'hrtfcnn',
        'hrtfcnn_repo_root': HRTFCNN_ROOT,
        'prepared_root': PREPARED_ROOT,
        'image_size': 64,
        'crop': {'x': 50, 'y': 50, 'width': 450, 'height': 450},
        'blur_kernel': 5,
        'edge_detection': {'enabled': True, 'low_threshold': 50, 'high_threshold': 100},
        'anthropometrics': {'use_first_n': 17},
    }
}

DATASET_CONFIG_PATH = '/tmp/dataset.yaml'
with open(DATASET_CONFIG_PATH, 'w') as f:
    yaml.dump(dataset_config, f)

print('Dataset config written.')

In [ ]:
!hrtf-personalization fetch-hrtfcnn-assets --config {DATASET_CONFIG_PATH}

## 3 — Prepare Training Samples

Extracts HRIRs and ear-image edge maps from each CIPIC subject into `.npz` files.

In [ ]:
!hrtf-personalization prepare-cipic --config {DATASET_CONFIG_PATH}

In [ ]:
import json
from pathlib import Path

manifest = json.loads(Path(f'{PREPARED_ROOT}/manifest.json').read_text())
print(f"Subjects : {manifest['num_subjects']}")
print(f"Samples  : {manifest['num_samples']}")

## 4 — Train the Conditional Model

Trains `ConditionalHRTFEstimator` for 150 epochs with direction and ear-side conditioning.

In [ ]:
CONDITIONAL_CHECKPOINT = f'{CHECKPOINT_DIR}/conditional.pt'

train_config = {
    'experiment': {'name': 'conditional-hrtfcnn'},
    'dataset': {
        'source_layout': 'hrtfcnn',
        'hrtfcnn_repo_root': HRTFCNN_ROOT,
        'prepared_root': PREPARED_ROOT,
        'image_size': 64,
        'crop': {'x': 50, 'y': 50, 'width': 450, 'height': 450},
        'blur_kernel': 5,
        'edge_detection': {'enabled': True, 'low_threshold': 50, 'high_threshold': 100},
        'anthropometrics': {'use_first_n': 17},
    },
    'training': {
        'batch_size': 64,
        'epochs': 150,
        'learning_rate': 1.0e-3,
        'seed': 7,
        'device': DEVICE,
        'optimizer': 'adam',
        'loss': 'mse',
        'log_interval_batches': 25,
    },
    'model': {
        'type': 'conditional',
        'anthropometric_dim': 17,
        'direction_dim': 2,
        'ear_side_dim': 1,
        'hrtf_dim': 200,
        'ear_image_size': 64,
    },
    'output': {'checkpoint': CONDITIONAL_CHECKPOINT},
}

TRAIN_CONFIG_PATH = '/tmp/train-conditional.yaml'
with open(TRAIN_CONFIG_PATH, 'w') as f:
    yaml.dump(train_config, f)

print('Training config written.')

In [ ]:
!hrtf-personalization train-conditional --config {TRAIN_CONFIG_PATH}

### Training Loss Curve

In [ ]:
import matplotlib.pyplot as plt

checkpoint = torch.load(CONDITIONAL_CHECKPOINT, map_location='cpu')
history = checkpoint['loss_history']

plt.figure(figsize=(8, 4))
plt.plot(history)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss')
plt.tight_layout()
plt.show()
print(f'Final loss: {history[-1]:.6f}')

## 5 — Evaluate

In [ ]:
METRICS_PATH = f'{REPO_DIR}/artifacts/reports/conditional_metrics.json'

eval_config = {
    'evaluation': {
        'checkpoint': CONDITIONAL_CHECKPOINT,
        'prepared_root': PREPARED_ROOT,
        'model_type': 'conditional',
        'batch_size': 64,
        'output_path': METRICS_PATH,
    }
}

EVAL_CONFIG_PATH = '/tmp/eval-conditional.yaml'
with open(EVAL_CONFIG_PATH, 'w') as f:
    yaml.dump(eval_config, f)

!hrtf-personalization evaluate --config {EVAL_CONFIG_PATH}

In [ ]:
metrics = json.loads(Path(METRICS_PATH).read_text())
print(f"Samples evaluated : {metrics['num_samples']}")
print(f"RMSE  mean +/- std  : {metrics['rmse_mean']:.4f} +/- {metrics['rmse_std']:.4f}")
print(f"LSD   mean +/- std  : {metrics['lsd_mean']:.4f} +/- {metrics['lsd_std']:.4f} dB")

## 6 — Predict Your Personalized SOFA

Upload a photo of each ear. The output is a `SimpleFreeFieldHRIR` SOFA file at 192 kHz.

In [ ]:
from google.colab import files

print('Upload your LEFT ear photo:')
left_upload = files.upload()
LEFT_IMAGE = f"/tmp/{list(left_upload.keys())[0]}"
with open(LEFT_IMAGE, 'wb') as f:
    f.write(list(left_upload.values())[0])

print('\nUpload your RIGHT ear photo:')
right_upload = files.upload()
RIGHT_IMAGE = f"/tmp/{list(right_upload.keys())[0]}"
with open(RIGHT_IMAGE, 'wb') as f:
    f.write(list(right_upload.values())[0])

print(f'\nLeft : {LEFT_IMAGE}')
print(f'Right: {RIGHT_IMAGE}')

In [ ]:
# Paste 17 comma-separated anthropometric values, or leave as None for defaults.
ANTHROPOMETRICS = None
# Example: '15.6,23.2,20.5,2.6,3.8,11.8,10.5,12.2,37.5,30.0,23.8,45.0,10.7,71.5,37.0,22.75,47.0'

TEMPLATE_SOFA = f'{REPO_DIR}/template.sofa'
OUTPUT_SOFA = '/tmp/my_hrtf.sofa'

anthro_flag = f'--anthro "{ANTHROPOMETRICS}"' if ANTHROPOMETRICS else ''

!hrtf-personalization predict \
    --checkpoint {CONDITIONAL_CHECKPOINT} \
    --left-image {LEFT_IMAGE} \
    --right-image {RIGHT_IMAGE} \
    --template-sofa {TEMPLATE_SOFA} \
    --output-sofa {OUTPUT_SOFA} \
    --device {DEVICE} \
    {anthro_flag}

## 7 — Inspect the Output SOFA

In [ ]:
import h5py
import numpy as np

with h5py.File(OUTPUT_SOFA, 'r') as f:
    ir  = np.asarray(f['Data.IR'])
    sr  = float(np.asarray(f['Data.SamplingRate'])[0])
    pos = np.asarray(f['SourcePosition'])

print(f'Sample rate : {sr:.0f} Hz')
print(f'Directions  : {ir.shape[0]}')
print(f'IR length   : {ir.shape[2]} samples ({ir.shape[2] / sr * 1000:.2f} ms)')
print(f'Left  peak  : {np.max(np.abs(ir[:, 0, :])):.4f}')
print(f'Right peak  : {np.max(np.abs(ir[:, 1, :])):.4f}')

ild = 20 * np.log10(
    (np.sqrt(np.mean(ir[:, 1, :]**2, axis=-1)) + 1e-12) /
    (np.sqrt(np.mean(ir[:, 0, :]**2, axis=-1)) + 1e-12)
)
print(f'Mean ILD    : {np.mean(ild):.2f} dB  (0 = balanced, + = right louder)')

In [ ]:
front_idx = int(np.argmin(np.abs(pos[:, 0]) + np.abs(pos[:, 1])))
t_ms = np.arange(ir.shape[2]) / sr * 1000

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for ax, ch, label in zip(axes, [0, 1], ['Left', 'Right']):
    ax.plot(t_ms, ir[front_idx, ch, :])
    ax.set_ylabel('Amplitude')
    ax.set_title(f'{label} ear — front (az={pos[front_idx,0]:.0f} deg, el={pos[front_idx,1]:.0f} deg)')
    ax.axhline(0, color='gray', linewidth=0.5)
axes[-1].set_xlabel('Time (ms)')
plt.tight_layout()
plt.show()

In [ ]:
freqs = np.fft.rfftfreq(ir.shape[2], d=1.0 / sr)

fig, ax = plt.subplots(figsize=(10, 4))
for ch, label, color in [(0, 'Left', 'steelblue'), (1, 'Right', 'tomato')]:
    mag_db = 20 * np.log10(np.abs(np.fft.rfft(ir[front_idx, ch, :])) + 1e-12)
    ax.semilogx(freqs[1:], mag_db[1:], label=label, color=color, alpha=0.85)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude (dB)')
ax.set_title('HRTF Magnitude — Front Direction')
ax.set_xlim(20, sr / 2)
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
files.download(OUTPUT_SOFA)